# 流式传输-概述

从智能体运行中流式实时更新

LangChain 实现了一个流式系统，用于实时更新。

流式传输对于提升基于大型语言模型（LLM）的应用程序响应性至关重要。通过逐步显示输出，甚至在完整响应准备就绪之前，流式传输显著提升了用户体验（UX），尤其是在处理LLM延迟问题时。

## 1. 概述
LangChain的流式传输系统允许将智能体运行的实时反馈呈现给应用。

LangChain 流式传输的可能性：
- 流代理进展——在每个智能体步骤后获取状态更新。
- 流式LLM tokens—— 在生成时流式语言模型tokens。
- 流式自定义更新—— 发送用户定义信号（例如，`"已获取 10/100 条记录"`）。
- 流式传输多种模式 — 从 `updates` (agent 进度)、`messages` (LLM token + 元数据) 或 `custom` (任意用户数据) 中选择。

## 2. 支持的流式传输模式
将以下一种或多种流式传输模式作为列表传递给`stream`或`astream`方法：
| 模式      | 描述                                                                 |
|:-----------|:----------------------------------------------------------------------|
| `updates`  | Streams 在每个智能体步骤后进行状态更新。如果同一步进行多次更新（例如运行多个节点），这些更新会分别流式传输。 |
| `messages` | 从调用 LLM 的任意图节点流出的元组 `(token, metadata)`。               |
| `custom`   | 通过流编写器从图节点内部流式传输自定义数据。                          |
## 3. Agent进度
要流式传输智能体进度，可以使用stream或astream方法 。每执行一个智能体步骤后都会发出一个事件。`stream_mode="updates"`

例如，如果有一个智能体只调用一次工具，应该会看到以下更新：
- LLM节点：AIMessage带有工具调用请求
- 工具节点：ToolMessage执行结果为
- LLM节点：最终AI响应

In [1]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'id': '77b08faf-d1f3-4065-b437-ecffff467b5b', 'name': 'get_weather', 'args': {'city': 'SF'}}]
step: tools
content: [{'type': 'text', 'text': "It's always sunny in SF!"}]
step: model
content: [{'type': 'text', 'text': "It's always sunny in SF! 🌤️  \nLet me know if you need more info!"}]


## 4. LLM Tokens
要流式传输 LLM 生成的 tokens，可以使用 `stream_mode="messages"`。可以在下面看到 agent 流式传输工具调用和最终响应的输出。

In [4]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
):
    if token.content_blocks:
        print(f"node: {metadata['langgraph_node']}")
        print(f"content: {token.content_blocks}")
        # print("\n")

node: model
content: [{'type': 'tool_call_chunk', 'id': '2ed39a80-a5ac-40d0-a587-c61047f0ef6e', 'name': 'get_weather', 'args': '{"city": "SF"}'}]
node: tools
content: [{'type': 'text', 'text': "It's always sunny in SF!"}]
node: model
content: [{'type': 'text', 'text': 'It'}]
node: model
content: [{'type': 'text', 'text': "'s"}]
node: model
content: [{'type': 'text', 'text': ' always'}]
node: model
content: [{'type': 'text', 'text': ' sunny'}]
node: model
content: [{'type': 'text', 'text': ' in'}]
node: model
content: [{'type': 'text', 'text': ' San'}]
node: model
content: [{'type': 'text', 'text': ' Francisco'}]
node: model
content: [{'type': 'text', 'text': '!'}]
node: model
content: [{'type': 'text', 'text': ' 🌞'}]
node: model
content: [{'type': 'text', 'text': ' Let'}]
node: model
content: [{'type': 'text', 'text': ' me'}]
node: model
content: [{'type': 'text', 'text': ' know'}]
node: model
content: [{'type': 'text', 'text': ' if'}]
node: model
content: [{'type': 'text', 'text': ' y

## 5. 自定义更新
要流式传输工具执行时的更新，可以使用 `get_stream_writer`。

In [5]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="custom"
):
    print(chunk)

Looking up data for city: San Francisco
Acquired data for city: San Francisco


## 6. 流式传输多种模式
可以通过将流模式作为列表传递来指定多种流模式：stream_mode=["updates", "custom"]

流式输出将是元组，其中 mode 是流模式的名称，chunk 是该模式流式传输的数据。

In [7]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for stream_mode, chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["updates", "custom"]
):
    print(f"stream_mode: {stream_mode}")
    print(f"content: {chunk}")
    # print("\n")

stream_mode: updates
content: {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-16T09:06:42.9775152Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2548656200, 'load_duration': 133110400, 'prompt_eval_count': 143, 'prompt_eval_duration': 43576400, 'eval_count': 102, 'eval_duration': 2343519100, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'}, id='lc_run--019bc60e-8aaa-7473-852a-50a63aa576f2-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'fa6c4e23-d377-4fff-a4e3-da93561666af', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 102, 'total_tokens': 245})]}}
stream_mode: custom
content: Looking up data for city: SF
stream_mode: custom
content: Acquired data for city: SF
stream_mode: updates
content: {'tools': {'messages': [ToolMessage(content="It's always sunny in SF!", name='get_weather', id='a83f09c

## 7. 常见模式
以下是展示流式传输常见用例的示例。
### 7.1 流式传输工具调用(Streaming Tools Calls)
可以同时流式传输两项内容：
- 工具调用生成过程中的部分 JSON 数据
- 经解析并待执行的完整工具调用指令

指定 `stream_mode="messages"` 时，将流式传输智能体中所有大语言模型调用生成的增量消息块。若要获取包含已解析工具调用的完整消息：

- 若这些消息在状态中被追踪（如 create_agent 的模型节点中），可通过 stream_mode=["messages", "updates"] 借助状态更新来获取完整消息（下文示例）。
- 若这些消息未在状态中被追踪，请使用自定义更新，或在流式传输循环中聚合消息块（详见下一节）。

In [8]:
from typing import Any

from langchain.agents import create_agent
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage, ToolMessage
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(model, tools=[get_weather])


def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)
    # N.B. all content is available through token.content_blocks


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")


input_message = {"role": "user", "content": "What is the weather in Boston?"}
for stream_mode, data in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],
):
    if stream_mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):  # `source` captures node name
                _render_completed_message(update["messages"][-1])

[{'name': 'get_weather', 'args': '{"city": "Boston"}', 'id': 'd76fd45e-0e34-4842-99c3-1cc9f7c951bd', 'index': None, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'd76fd45e-0e34-4842-99c3-1cc9f7c951bd', 'type': 'tool_call'}]
Tool response: [{'type': 'text', 'text': "It's always sunny in Boston!"}]
It|'s| always| sunny| in| Boston|!| 🌤|️|

### 7.2 获取已完成的消息
在某些情况下，已完成的消息不会体现在状态更新中。若有权限访问智能体的内部机制，可在流式传输过程中通过自定义更新来获取这些消息。否则，可以在流式循环中聚合消息分片（见下文）。 ？？？

请参考以下示例：在一个简化的安全校验中间件中集成了流式写入器。该中间件通过工具调用生成结构化的 “安全 / 不安全” 评估结果（也可借助结构化输出功能实现此需求）。

In [15]:
from typing import Any, Literal

from langchain.agents.middleware import after_agent, AgentState
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from langchain.chat_models import init_chat_model
from langgraph.config import get_stream_writer
from pydantic import BaseModel
from langchain_ollama import ChatOllama


class ResponseSafety(BaseModel):
    """Evaluate a response as safe or unsafe."""
    evaluation: Literal["safe", "unsafe"]

# safety_model = init_chat_model("openai:gpt-5.2")
safety_model = ChatOllama(model="qwen3:4b")

@after_agent(can_jump_to=["end"])
def safety_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Model-based guardrail: Use an LLM to evaluate response safety."""
    stream_writer = get_stream_writer()
    # Get the model response
    if not state["messages"]:
        return None

    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return None

    # Use another model to evaluate safety
    model_with_tools = safety_model.bind_tools([ResponseSafety], tool_choice="any")
    result = model_with_tools.invoke(
        [
            {
                "role": "system",
                "content": "Evaluate this AI response as generally safe or unsafe."
            },
            {
                "role": "user",
                "content": f"AI response: {last_message.text}"
            }
        ]
    )
    stream_writer(result)

    tool_call = result.tool_calls[0]
    if tool_call["args"]["evaluation"] == "unsafe":
        last_message.content = "I cannot provide that response. Please rephrase your request."

    return None

随后，即可将该中间件集成至智能体中，并纳入其自定义的流式事件。

In [12]:
from typing import Any

from langchain.agents import create_agent
from langchain.messages import AIMessageChunk, AIMessage, AnyMessage, ToolMessage

model = init_chat_model("ollama:qwen3:0.6b")

def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[safety_guardrail],
)

def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")


input_message = {"role": "user", "content": "What is the weather in Boston?"}
for stream_mode, data in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates", "custom"],
):
    if stream_mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])
    if stream_mode == "custom":
        # access completed message in stream
        print(f"Tool calls: {data.tool_calls}")

[{'name': 'get_weather', 'args': '{"city": "Boston"}', 'id': '91bbcbfa-44ea-4412-8ef9-091310a0211f', 'index': None, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': '91bbcbfa-44ea-4412-8ef9-091310a0211f', 'type': 'tool_call'}]
Tool response: [{'type': 'text', 'text': "It's always sunny in Boston!"}]
It|'s| always| sunny| in| Boston|!|[{'name': 'ResponseSafety', 'args': '{"evaluation": "unsafe"}', 'id': '9fd6aee2-31eb-4579-ab47-c819c89aa59c', 'index': None, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'ResponseSafety', 'args': {'evaluation': 'unsafe'}, 'id': '9fd6aee2-31eb-4579-ab47-c819c89aa59c', 'type': 'tool_call'}]


反之，若无法在流中添加自定义事件，则可在流式循环内聚合消息分片。

In [16]:
input_message = {"role": "user", "content": "What is the weather in Boston?"}
full_message = None
for stream_mode, data in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],
):
    if stream_mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
            full_message = token if full_message is None else full_message + token
            if token.chunk_position == "last":
                if full_message.tool_calls:
                    print(f"Tool calls: {full_message.tool_calls}")
                full_message = None
    if stream_mode == "updates":
        for source, update in data.items():
            if source == "tools":
                _render_completed_message(update["messages"][-1])

[{'name': 'ResponseSafety', 'args': '{"evaluation": "unsafe"}', 'id': '9294b49c-32d6-40c9-ae0a-a99ca6063bd5', 'index': None, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'ResponseSafety', 'args': {'evaluation': 'unsafe'}, 'id': '9294b49c-32d6-40c9-ae0a-a99ca6063bd5', 'type': 'tool_call'}]


### 7.3 人机协同流式处理
若要实现人机协同中断功能，可基于上述示例进行扩展：
- 为智能体配置人机协同中间件与检查点工具
- 收集在 “更新” 流模式下产生的中断信息
- 依据指令响应该类中断

In [17]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, Interrupt


def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"


checkpointer = InMemorySaver()

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"get_weather": True}),
    ],
    checkpointer=checkpointer,
)


def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")


def _render_interrupt(interrupt: Interrupt) -> None:
    interrupts = interrupt.value
    for request in interrupts["action_requests"]:
        print(request["description"])


input_message = {
    "role": "user",
    "content": (
        "Can you look up the weather in Boston and San Francisco?"
    ),
}
config = {"configurable": {"thread_id": "some_id"}}
interrupts = []
for stream_mode, data in agent.stream(
    {"messages": [input_message]},
    config=config,
    stream_mode=["messages", "updates"],
):
    if stream_mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])
            if source == "__interrupt__":
                interrupts.extend(update)
                _render_interrupt(update[0])

[{'name': 'get_weather', 'args': '{"city": "Boston"}', 'id': '684f9099-8bf1-459d-9399-cfaa4533db77', 'index': None, 'type': 'tool_call_chunk'}]
[{'name': 'get_weather', 'args': '{"city": "San Francisco"}', 'id': '5539e1b2-d7ad-409d-bd2a-d07ec657961a', 'index': None, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': '684f9099-8bf1-459d-9399-cfaa4533db77', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '5539e1b2-d7ad-409d-bd2a-d07ec657961a', 'type': 'tool_call'}]
Tool execution requires approval

Tool: get_weather
Args: {'city': 'Boston'}
Tool execution requires approval

Tool: get_weather
Args: {'city': 'San Francisco'}


接下来，为每一次中断收集对应的决策指令。重点注意，决策指令的顺序必须与收集到的操作顺序保持一致。
为便于说明，将编辑其中一个工具调用指令，并接受另一个工具调用指令。

In [18]:
def _get_interrupt_decisions(interrupt: Interrupt) -> list[dict]:
    return [
        {
            "type": "edit",
            "edited_action": {
                "name": "get_weather",
                "args": {"city": "Boston, U.K."},
            },
        }
        if "boston" in request["description"].lower()
        else {"type": "approve"}
        for request in interrupt.value["action_requests"]
    ]

decisions = {}
for interrupt in interrupts:
    decisions[interrupt.id] = {
        "decisions": _get_interrupt_decisions(interrupt)
    }

decisions

{'36bb191cb5e2b9c6ef95793d13ff333c': {'decisions': [{'type': 'edit',
    'edited_action': {'name': 'get_weather',
     'args': {'city': 'Boston, U.K.'}}},
   {'type': 'approve'}]}}

只需向同一个流式循环传入一条指令，即可恢复运行。

In [19]:
interrupts = []
for stream_mode, data in agent.stream(
    Command(resume=decisions),
    config=config,
    stream_mode=["messages", "updates"],
):
    # Streaming loop is unchanged
    if stream_mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])
            if source == "__interrupt__":
                interrupts.extend(update)
                _render_interrupt(update[0])

Tool response: [{'type': 'text', 'text': "It's always sunny in Boston, U.K.!"}]
Tool response: [{'type': 'text', 'text': "It's always sunny in San Francisco!"}]
It|'s| always| sunny| in| Boston|,| U|.K|.|!|  
|It|'s| always| sunny| in| San| Francisco|!| 🌞|

### 7.4 子智能体流式输出
当智能体在任意节点包含多个大语言模型时，往往需要在消息生成的同时明确区分其来源。

要实现这一功能，可在创建每个智能体时为其分配一个名称。在 **“消息” 模式 ** 下进行流式传输时，该名称可通过元数据中的 lc_agent_name 键获取。

下文将对工具调用流式输出示例进行更新：
- 将原有工具替换为 call_weather_agent 工具，该工具会在内部调用一个智能体
- 为每个智能体添加专属名称
- 创建流时，将参数 subgraphs 设为 True

流的处理逻辑与之前保持一致，新增逻辑为：利用创建智能体时传入的 name 参数，跟踪当前处于活跃状态的智能体。

In [21]:
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, AnyMessage
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"


# weather_model = init_chat_model("openai:gpt-5.2")
weather_model = ChatOllama(model="qwen3:0.6b")
weather_agent = create_agent(
    model=weather_model,
    tools=[get_weather],
    name="weather_agent",
)


def call_weather_agent(query: str) -> str:
    """Query the weather agent."""
    result = weather_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].text


# supervisor_model = init_chat_model("openai:gpt-5.2")
supervisor_model = ChatOllama(model="qwen3:1.7b")
agent = create_agent(
    model=supervisor_model,
    tools=[call_weather_agent],
    name="supervisor",
)

In [22]:
def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")


input_message = {"role": "user", "content": "What is the weather in Boston?"}
current_agent = None
for _, stream_mode, data in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],
    subgraphs=True,
):
    if stream_mode == "messages":
        token, metadata = data
        if agent_name := metadata.get("lc_agent_name"):
            if agent_name != current_agent:
                print(f"🤖 {agent_name}: ")
                current_agent = agent_name
        if isinstance(token, AIMessage):
            _render_message_chunk(token)
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])

[{'name': 'call_weather_agent', 'args': '{"query": "What is the weather in Boston?"}', 'id': '501fb9d2-9baf-496c-af4a-722374152555', 'index': None, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'call_weather_agent', 'args': {'query': 'What is the weather in Boston?'}, 'id': '501fb9d2-9baf-496c-af4a-722374152555', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': '{"city": "Boston"}', 'id': 'a3762523-874b-43bc-a37f-35f967fe7353', 'index': None, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'a3762523-874b-43bc-a37f-35f967fe7353', 'type': 'tool_call'}]
Tool response: [{'type': 'text', 'text': "It's always sunny in Boston!"}]
It|'s| always| sunny| in| Boston|!| 🌞|Tool response: [{'type': 'text', 'text': "It's always sunny in Boston! 🌞"}]
The| weather| in| Boston| is| described| as| always| sunny|!| 🌞| Let| me| know| if| you|'d| like| current| updates| or| other| details|!|

## 8. 关闭流式传输
在部分应用场景中，可能需要针对指定模型关闭逐令牌流式输出功能。该操作在以下场景中尤为实用：

构建多智能体系统时，控制哪些智能体开启流式输出

混用支持流式传输与不支持流式传输的模型

部署至 LangSmith 平台时，避免特定模型的输出内容流式推送至客户端

初始化模型时，将参数 streaming 设为 False 即可完成配置。

In [24]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4o",
    streaming=False
)